In [1]:
"""
Sample code to visualize stock price data using Plotly.

This demonstrates different chart types for analyzing price data:
- Candlestick chart (most common for stock data)
- Line chart with volume
- OHLC chart
- Interactive chart with range selector
"""

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Load the data
project_root = Path.cwd().parent
data_path = project_root / "data" / "test_data.csv"

df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sort by timestamp to ensure proper ordering
df = df.sort_values('timestamp')

print(f"Loaded {len(df)} rows of data for {df['symbol'].unique()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Loaded 1004 rows of data for ['AAPL' 'GOOGL' 'TSLA' 'MSFT']
Date range: 2024-01-02 05:00:00+00:00 to 2024-12-30 05:00:00+00:00


# Stock Price Visualization Examples

This notebook demonstrates various ways to visualize stock price data using Plotly.

## Chart 1: Candlestick Chart with Volume

The classic financial chart showing OHLC data with volume bars below.

In [3]:
def create_candlestick_with_volume(df):
    """Create a candlestick chart with volume subplot."""

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=('AAPL Price', 'Volume'),
        row_heights=[0.7, 0.3]
    )

    # Add candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=df['timestamp'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='AAPL',
            increasing_line_color='green',
            decreasing_line_color='red'
        ),
        row=1, col=1
    )

    # Add volume bars
    colors = ['green' if close >= open else 'red'
              for close, open in zip(df['close'], df['open'])]

    fig.add_trace(
        go.Bar(
            x=df['timestamp'],
            y=df['volume'],
            name='Volume',
            marker_color=colors,
            showlegend=False
        ),
        row=2, col=1
    )

    # Update layout
    fig.update_layout(
        title='AAPL Stock Price and Volume',
        yaxis_title='Price ($)',
        yaxis2_title='Volume',
        xaxis_rangeslider_visible=False,
        height=800,
        template='plotly_dark'
    )

    return fig

# Generate and show the chart
fig1 = create_candlestick_with_volume(df.copy())
fig1.show()

## Chart 2: Line Chart with Moving Averages

Shows the closing price with 10-day and 20-day moving averages.

In [4]:
def create_line_chart_with_ma(df):
    """Create a line chart with moving averages."""

    # Calculate moving averages
    df['MA10'] = df['close'].rolling(window=10).mean()
    df['MA20'] = df['close'].rolling(window=20).mean()

    fig = go.Figure()

    # Add closing price
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['close'],
            name='Close Price',
            line=dict(color='lightblue', width=2)
        )
    )

    # Add 10-day MA
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['MA10'],
            name='10-Day MA',
            line=dict(color='orange', width=1.5, dash='dash')
        )
    )

    # Add 20-day MA
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['MA20'],
            name='20-Day MA',
            line=dict(color='red', width=1.5, dash='dash')
        )
    )

    fig.update_layout(
        title='AAPL Close Price with Moving Averages',
        xaxis_title='Date',
        yaxis_title='Price ($)',
        height=600,
        template='plotly_dark',
        hovermode='x unified'
    )

    return fig

# Generate and show the chart
fig2 = create_line_chart_with_ma(df.copy())
fig2.show()

## Chart 3: OHLC Chart with VWAP

Shows Open-High-Low-Close bars with Volume-Weighted Average Price overlay.

In [5]:
def create_ohlc_with_vwap(df):
    """Create an OHLC chart with VWAP overlay."""

    fig = go.Figure()

    # Add OHLC chart
    fig.add_trace(
        go.Ohlc(
            x=df['timestamp'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='OHLC',
            increasing_line_color='green',
            decreasing_line_color='red'
        )
    )

    # Add VWAP
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['vwap'],
            name='VWAP',
            line=dict(color='yellow', width=2)
        )
    )

    fig.update_layout(
        title='AAPL OHLC with VWAP',
        xaxis_title='Date',
        yaxis_title='Price ($)',
        height=600,
        template='plotly_dark'
    )

    return fig

# Generate and show the chart
fig3 = create_ohlc_with_vwap(df.copy())
fig3.show()

## Chart 4: Interactive Dashboard

A comprehensive dashboard with range selector buttons for zooming to different time periods.

In [6]:
def create_interactive_chart(df):
    """Create an interactive chart with range selector and buttons."""

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=('Price', 'Volume', 'VWAP vs Close'),
        row_heights=[0.5, 0.25, 0.25]
    )

    # Add candlestick
    fig.add_trace(
        go.Candlestick(
            x=df['timestamp'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='Price'
        ),
        row=1, col=1
    )

    # Add volume
    colors = ['green' if close >= open else 'red'
              for close, open in zip(df['close'], df['open'])]

    fig.add_trace(
        go.Bar(
            x=df['timestamp'],
            y=df['volume'],
            name='Volume',
            marker_color=colors
        ),
        row=2, col=1
    )

    # Add VWAP comparison
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['close'],
            name='Close',
            line=dict(color='lightblue')
        ),
        row=3, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['vwap'],
            name='VWAP',
            line=dict(color='yellow')
        ),
        row=3, col=1
    )

    # Add range selector
    fig.update_xaxes(
        rangeslider_visible=False,
        rangeselector=dict(
            buttons=list([
                dict(count=7, label="1w", step="day", stepmode="backward"),
                dict(count=14, label="2w", step="day", stepmode="backward"),
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=3, label="3m", step="month", stepmode="backward"),
                dict(step="all", label="All")
            ])
        ),
        row=3, col=1
    )

    fig.update_layout(
        title='AAPL Interactive Analysis Dashboard',
        height=1000,
        template='plotly_dark',
        showlegend=True
    )

    return fig

# Generate and show the chart
fig4 = create_interactive_chart(df.copy())
fig4.show()

## Chart 5: Returns Analysis

Shows daily returns over time and their distribution.

In [7]:
def create_returns_analysis(df):
    """Create returns analysis charts."""

    # Calculate daily returns
    df['returns'] = df['close'].pct_change() * 100

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Daily Returns Over Time', 'Returns Distribution'),
        specs=[[{"type": "scatter"}, {"type": "histogram"}]]
    )

    # Returns over time
    colors = ['green' if r > 0 else 'red' for r in df['returns']]

    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['returns'],
            mode='markers',
            name='Daily Returns',
            marker=dict(color=colors, size=6)
        ),
        row=1, col=1
    )

    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

    # Returns distribution
    fig.add_trace(
        go.Histogram(
            x=df['returns'].dropna(),
            name='Distribution',
            nbinsx=30,
            marker_color='lightblue'
        ),
        row=1, col=2
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Returns (%)", row=1, col=2)
    fig.update_yaxes(title_text="Returns (%)", row=1, col=1)
    fig.update_yaxes(title_text="Frequency", row=1, col=2)

    fig.update_layout(
        title='AAPL Returns Analysis',
        height=500,
        template='plotly_dark',
        showlegend=False
    )

    return fig

# Generate and show the chart
fig5 = create_returns_analysis(df.copy())
fig5.show()